# Quick Start: Virtual Knockout Analysis for Erlotinib

This notebook demonstrates how to run virtual knockout analysis to identify key pathways and genes that contribute to drug sensitivity vs resistance.

**Expected runtime:** ~5 minutes

## What is Virtual Knockout?
Virtual knockout analysis identifies which pathways and genes causally affect drug response by:
1. Identifying top 20 sensitive and bottom 20 resistant cell lines
2. Virtually "knocking out" each pathway/gene in the model
3. Measuring the impact on predicted drug sensitivity
4. Identifying pathways/genes that drive sensitivity vs resistance

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from src.model_interpretability.model_pathway_and_gene_interpretability import (
    load_config,
    main as run_knockout_analysis
)

## Setup: Load Model and Configuration

In [2]:
# Drug to analyze
drug_name = "Erlotinib"

# Load configuration (using notebook-specific config with correct paths)
config_path = "configs/config_knockout.yaml"
config = load_config(config_path)

# Set number of cell lines to analyze (top/bottom 20)
top_n = 20
config['top_n_cell_lines'] = top_n

print(f"Analyzing drug: {drug_name}")
print(f"Top/bottom cell lines: {top_n}")
print(f"Output directory: {config['output_dir']}")

Analyzing drug: Erlotinib
Top/bottom cell lines: 20
Output directory: ../data/output_data/quickstart/interpretability


## Identify Top Sensitive and Resistant Cell Lines

In [3]:
# Load drug response data
response_file = Path("../data/intermediate_data/CTRPv2_drug_response_data") / f"{drug_name}.csv"
df = pd.read_csv(response_file)

# Sort by AAC (higher = more sensitive)
df_sorted = df.sort_values('aac', ascending=False)

# Get top sensitive and resistant cell lines
top_sensitive = df_sorted.head(top_n)['ModelID'].tolist()
top_resistant = df_sorted.tail(top_n)['ModelID'].tolist()

print(f"Top {top_n} sensitive cell lines identified")
print(f"Top {top_n} resistant cell lines identified")
print(f"\nExample sensitive cell lines: {top_sensitive[:5]}")
print(f"Example resistant cell lines: {top_resistant[:5]}")

Top 20 sensitive cell lines identified
Top 20 resistant cell lines identified

Example sensitive cell lines: ['ACH-000030', 'ACH-000489', 'ACH-000741', 'ACH-000723', 'ACH-000066']
Example resistant cell lines: ['ACH-000050', 'ACH-000363', 'ACH-000402', 'ACH-000743', 'ACH-000514']


## Run Pathway Knockout Analysis

In [ ]:
# Run pathway knockout
print("Running pathway knockout analysis...")
print("This will identify which pathways drive sensitivity vs resistance")

# Modify config for pathway knockout
config['knockout_target'] = 'pathway'
config['drugs_to_process'] = [drug_name]

# Run knockout analysis
pathway_results = run_knockout_analysis(config, [drug_name])

print("\nPathway knockout analysis completed!")
print(f"Results saved to: {config['output_dir']}/SinglePathwayKO/{drug_name}/")

Running pathway knockout analysis...
This will identify which pathways drive sensitivity vs resistance
Mode: PATHWAY KNOCKOUT
Loaded relevant entities from: ../src/model_interpretability/relevant_entities/relevant_genes.json
Loading GO term names from: ../data/input_data/all_pathway_genesets.gmt
Loaded 235 GO term mappings
Per-drug mode: Model path is a directory - ../data/output_data/quickstart/
Global data loaded
Loaded gene minimum RNA map from cache with 19138 entries

Loading per-drug model for Erlotinib: ../data/output_data/quickstart/Erlotinib/Erlotinib_20260107_135351/model_runfixed_params_seed42_best_fold1_e9_selected.pth
Loading checkpoint: ../data/output_data/quickstart/Erlotinib/Erlotinib_20260107_135351/model_runfixed_params_seed42_best_fold1_e9_selected.pth
Omics type: all
Loaded pathway graph: 235 nodes, 2030 edges
Graph contains cycles (feedback loops) - using iterative message passing
Initialized DrugEmbedderANN: 256 -> 1024 -> 512 -> 128 | Dropout: 0.4

==============

Selecting top/bottom 20 cell lines by AAC
Running knockout on 40 samples


Drug-specific 95th percentile threshold: 0.0097
Summary saved: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/summary/Erlotinib_gdsc0_true_test_knockout_summary.csv
Raw scores saved: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/raw_scores/Erlotinib_gdsc0_true_test_knockout_raw_scores.csv

Generating heatmaps for Erlotinib in gdsc0_true_test
Loaded cell line name map
Using Measured AAC for sorting
Including Measured AAC in heatmap color bar
Generating heatmaps sorted by_mean_importance
Heatmap data saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/plots/heatmaps_by_mean_importance/01_no_mask/csv/Erlotinib_gdsc0_true_test_all_samples_heatmap.csv
Small plot detected (50x40), using DPI=200
Heatmap saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/plots/heatmaps_by_mean_importance/01_no_mask/Erlotinib_gdsc0_true_test_all_samples_heatmap.png
Gene

/home/charif/PIGE/PIGE/src/model_interpretability/plotting_interpretability.py:408: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Bottom Quartile (Resistant)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  plot_data.loc[plot_data[sensitivity_col] <= lower_quartile, 'response_group'] = 'Bottom Quartile (Resistant)'


Quartile box plot saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/plots/box_plots_quartile/predicted/Erlotinib_regulation_of_MAPK1_and_MAPK3_cascade_quartile_boxplot.png
Top/bottom responder box plot saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/plots/box_plots_top_bottom_10/predicted/Erlotinib_regulation_of_MAPK1_and_MAPK3_cascade_top_bottom_10_boxplot.png
Raincloud plot of importance scores saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/plots/raincloud_plots/predicted/Erlotinib_regulation_of_MAPK1_and_MAPK3_cascade_raincloud.png
Scatter plot saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/plots/scatter_plots/predicted/Erlotinib_regulation_of_PI3K_signal_transduction_scatter.png


...

/home/charif/PIGE/PIGE/src/model_interpretability/plotting_interpretability.py:776: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0.03, 1, 0.95])


Summary heatmap saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/all_drugs_summary_differential_importance_heatmap.png
Heatmap data saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/csv_summary/all_drugs_summary_spearman_corr_actual.csv


/home/charif/PIGE/PIGE/src/model_interpretability/plotting_interpretability.py:776: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0.03, 1, 0.95])


Summary heatmap saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/all_drugs_summary_spearman_corr_actual_heatmap.png

Pathway knockout analysis completed!
Results saved to: ../data/output_data/quickstart/interpretability/SinglePathwayKO/Erlotinib/


: 

## Run Gene Knockout Analysis

In [ ]:
# Run gene knockout
print("Running gene knockout analysis...")
print("This will identify which genes drive sensitivity vs resistance")

# Modify config for gene knockout
config['knockout_target'] = 'gene'

# Run knockout analysis
gene_results = run_knockout_analysis(config, [drug_name])

print("\nGene knockout analysis completed!")
print(f"Results saved to: {config['output_dir']}/SingleGeneKO/{drug_name}/")

Running gene knockout analysis...
This will identify which genes drive sensitivity vs resistance
Mode: GENE KNOCKOUT
Loaded relevant entities from: ../src/model_interpretability/relevant_entities/relevant_genes.json
Loading GO term names from: ../data/input_data/all_pathway_genesets.gmt
Loaded 235 GO term mappings
Per-drug mode: Model path is a directory - ../data/output_data/quickstart/
Global data loaded
Loaded gene minimum RNA map from cache with 19138 entries

Loading per-drug model for Erlotinib: ../data/output_data/quickstart/Erlotinib/Erlotinib_20260107_135351/model_runfixed_params_seed42_best_fold1_e9_selected.pth
Loading checkpoint: ../data/output_data/quickstart/Erlotinib/Erlotinib_20260107_135351/model_runfixed_params_seed42_best_fold1_e9_selected.pth
Omics type: all
Loaded pathway graph: 235 nodes, 2030 edges
Graph contains cycles (feedback loops) - using iterative message passing
Initialized DrugEmbedderANN: 256 -> 1024 -> 512 -> 128 | Dropout: 0.4

==================== Pr

Selecting top/bottom 20 cell lines by AAC
Running knockout on 40 samples


Knockout for gdsc0_true_test:   0%|          | 0/40 [00:00<?, ?it/s]

## Run Edge Knockout Analysis

In [ ]:
# Run edge knockout
print("Running edge knockout analysis...")
print("This will identify which edges drive sensitivity vs resistance")

# Modify config for edge knockout
config['knockout_target'] = 'double_pathway'

# Run knockout analysis
edge_results = run_knockout_analysis(config, [drug_name])

print("\nEdge knockout analysis completed!")
print(f"Results saved to: {config['output_dir']}/DoublePathwayKO/{drug_name}/")

Running edge knockout analysis...
This will identify which edges drive sensitivity vs resistance
Mode: DOUBLE PATHWAY KNOCKOUT
Loaded relevant entities from: ../src/model_interpretability/relevant_entities/relevant_genes.json
Loading GO term names from: ../data/input_data/all_pathway_genesets.gmt
Loaded 235 GO term mappings
Per-drug mode: Model path is a directory - ../data/output_data/quickstart/
Global data loaded
Loaded gene minimum RNA map from cache with 19138 entries
No model found for drug 'Erlotinib' in ../data/output_data/quickstart/Erlotinib
Skipping Erlotinib - no model found


All drugs processed

Edge knockout analysis completed!
Results saved to: ../data/output_data/quickstart/interpretability/DoublePathwayKO/Erlotinib/


## View Results Summary

In [ ]:
# Display summary of key findings
output_dir = Path(config['output_dir'])

print("\n" + "="*60)
print("Virtual Knockout Analysis Summary")
print("="*60)
print(f"\nDrug: {drug_name}")
print(f"Cell lines analyzed: {top_n} sensitive + {top_n} resistant")
print(f"\nOutput files:")
print(f"  - Pathway results: {output_dir}/SinglePathwayKO/gdsc0_true_test/plots/heatmaps_by_mean_importance")
print(f"  - Gene results: {output_dir}/SingleGeneKO/gdsc0_true_test/plots/heatmaps_by_mean_importance")
print(f"  - Edge results: {output_dir}/DoublePathwayKO/gdsc0_true_test/plots/heatmaps_by_mean_importance")
print(f"\nNext step: Run 03_generate_pige_graphs.ipynb to visualize pathway crosstalk")


Virtual Knockout Analysis Summary

Drug: Erlotinib
Cell lines analyzed: 20 sensitive + 20 resistant

Output files:
  - Pathway results: ../data/output_data/quickstart/interpretability/SinglePathwayKO/gdsc0_true_test/plots/heatmaps_by_mean_importance
  - Gene results: ../data/output_data/quickstart/interpretability/SingleGeneKO/gdsc0_true_test/plots/heatmaps_by_mean_importance
  - Edge results: ../data/output_data/quickstart/interpretability/DoublePathwayKO/gdsc0_true_test/plots/heatmaps_by_mean_importance

Next step: Run 03_generate_pige_graphs.ipynb to visualize pathway crosstalk
